# 01 — Data Preprocessing

Load -> Duplicate handling -> Clean -> Preliminary split (70/30) -> MICE

**Duplicate logic**:
- Exact duplicates (same features + same targets, confirmed by ≥2 valid values): keep first, remove rest
- Replicate measurements (same features + different targets, confirmed by std ≥ 1e-6): take mean
- Indeterminate groups (all targets have <2 valid values, can't classify): keep all rows as-is

**Key design**:
- Preliminary 70/30 split (seed=42) for pipeline development — MICE fitted on train only
- Train set: MICE imputed (fit+transform) → N02 generates `X_train.npy` / `y_train.npy`
- Holdout set: MICE imputed (transform only) → N02 generates `X_test.npy` / `y_test.npy`
- Full cleaned dataset (NaN preserved) → N02 generates `X.npy` / `y.npy`; N04 Phase B does per-split MICE
- Imputer TYPE selected via cv_rmse on train, saved for N04 per-split MICE

**Input**: `data/Wrought_Al_alloy.xlsx`
**Output**: `data/metal_train_imputed.csv`, `data/metal_holdout_imputed.csv`, `data/metal_cleaned.csv`, `data/best_imputer.json`

In [1]:
import os, json, warnings, numpy as np, pandas as pd
from collections import OrderedDict
from time import time
warnings.filterwarnings('ignore')

RNG_SEED = 42; np.random.seed(RNG_SEED)
DATA_DIR = os.path.join('..', 'data')
os.makedirs(DATA_DIR, exist_ok=True)

MASS_FRAC_COLS = ['Si','Fe','Cu','Mn','Mg','Cr','Zn','V','Ti','Zr','Li','Ni','Be','Sc']
PROCESS_COLS   = ['Tsol','Tage','tage']
TARGET_COLS    = ['YS','UTS','El']
FEATURE_COLS   = MASS_FRAC_COLS + PROCESS_COLS

## 1. Load & Convert

In [2]:
RAW_PATH = os.path.join('..', 'data', 'Wrought_Al_alloy.xlsx')
df_raw = pd.read_excel(RAW_PATH)

# Drop truly useless columns (empty or nearly empty)
useless_cols = [c for c in df_raw.columns if c.startswith('Unnamed')]
if useless_cols:
    df_raw = df_raw.drop(columns=useless_cols)
    print(f'Dropped useless columns: {useless_cols}')

# Fix European decimal format: "4,2" -> 4.2 (comma as decimal separator)
# Some Excel files use commas; convert all object columns to proper numeric
for col in df_raw.columns:
    if df_raw[col].dtype == object:
        # Replace comma with dot, then try numeric conversion
        converted = df_raw[col].astype(str).str.replace(',', '.').str.strip()
        df_raw[col] = pd.to_numeric(converted, errors='ignore')

# ── Validate mass fraction columns are actually numeric ──
# errors='ignore' above silently leaves unconvertible columns as object;
# downstream division by 100 would then fail with a cryptic error.
non_numeric_mass = [c for c in MASS_FRAC_COLS if df_raw[c].dtype == object]
if non_numeric_mass:
    print(f'WARNING: {len(non_numeric_mass)} mass fraction column(s) still object-type '
          f'after decimal conversion: {non_numeric_mass}')
    print('  Attempting coercive conversion (non-numeric → NaN)...')
    for c in non_numeric_mass:
        # Re-apply conversion with coerce this time
        converted = df_raw[c].astype(str).str.replace(',', '.').str.strip()
        df_raw[c] = pd.to_numeric(converted, errors='coerce')
    na_added = df_raw[non_numeric_mass].isna().sum().sum()
    print(f'  Coerced {na_added} unparseable value(s) to NaN in mass fraction columns')

print(f'Loaded: {df_raw.shape[0]} rows x {df_raw.shape[1]} cols')

Dropped useless columns: ['Unnamed: 22', 'Unnamed: 23']
Loaded: 541 rows x 22 cols


In [3]:
# Convert wt% to fractions (divide by 100)
df_raw[MASS_FRAC_COLS] = df_raw[MASS_FRAC_COLS] / 100.0

s = df_raw[MASS_FRAC_COLS].sum(axis=1)
print(f'Solute sum (fraction): {s.min():.3f} ~ {s.max():.3f}')
print(f'Al balance = 100 - solute_sum: {(1-s.mean())*100:.1f}% on average')

Solute sum (fraction): 0.009 ~ 0.176
Al balance = 100 - solute_sum: 91.6% on average


## 2. Handle Duplicates (by composition + process)

In [4]:
# Group by (composition + process) to detect duplicates
# dropna=False: keep rows with NaN features, they get properly dropped in Step 3
grouped = df_raw.groupby(FEATURE_COLS, dropna=False)

exact_dup_pairs = []     # (kept_row_idx, removed_row_idx)
replicate_groups = []    # (row_indices_list, target_std_per_target)
indeterminate_groups = [] # groups where we lack enough target info to classify
merged_rows = []

for key, group in grouped:
    if len(group) == 1:
        merged_rows.append(group.iloc[0])
        continue

    idx_list = group.index.tolist()

    # Force float to avoid np.isnan TypeError with mixed dtypes
    target_vals = group[TARGET_COLS].astype(float).values
    target_std = np.full(len(TARGET_COLS), np.nan)  # NaN = can't judge

    for i in range(len(TARGET_COLS)):
        vals = target_vals[:, i]
        valid = ~np.isnan(vals)
        n_valid = valid.sum()
        if n_valid >= 2:
            target_std[i] = np.std(vals[valid])
        # n_valid <= 1 → leave as NaN (insufficient data to judge spread)

    # Classify: need at least one target with ≥2 valid values to draw a conclusion
    any_variation = np.any(target_std >= 1e-6)           # NaN compares → False
    any_confirmed_identical = np.any((~np.isnan(target_std)) & (target_std < 1e-6))

    if any_variation:
        # Confirmed replicate measurements: same features, different targets. Take mean.
        replicate_groups.append((idx_list, target_std))
        row = group.iloc[0].copy()
        row[TARGET_COLS] = group[TARGET_COLS].astype(float).mean().values
        if 'Reference (APA)' in group.columns:
            refs = group['Reference (APA)'].dropna().unique()
            if len(refs) > 3:
                print(f'  NOTE: {len(refs)} references truncated to 3 for rows {idx_list}')
            row['Reference (APA)'] = ' | '.join(refs[:3])
        merged_rows.append(row)

    elif any_confirmed_identical:
        # Confirmed exact duplicate: same features + same targets. Keep first.
        for removed_idx in idx_list[1:]:
            exact_dup_pairs.append((idx_list[0], removed_idx))
        merged_rows.append(group.iloc[0])

    else:
        # Indeterminate: all targets have < 2 valid values across the group.
        # We cannot tell if these are duplicates or distinct measurements.
        # SAFEST PATH: keep ALL rows (each may carry unique metadata / Reference).
        # Targets stay as-is (NaN preserved); MICE will impute later.
        indeterminate_groups.append(idx_list)
        for _, row in group.iterrows():
            merged_rows.append(row.copy())

df = pd.DataFrame(merged_rows).reset_index(drop=True)

print(f'Rows after merging: {len(df)} (was {len(df_raw)})')
print(f'Exact duplicates removed: {len(exact_dup_pairs)} rows')
if exact_dup_pairs:
    for kept, removed in exact_dup_pairs[:5]:
        print(f'  Row {kept} -> removed Row {removed}')
    if len(exact_dup_pairs) > 5:
        print(f'  ... and {len(exact_dup_pairs)-5} more')
print(f'Replicate groups merged (mean of targets): {len(replicate_groups)} groups')
if replicate_groups:
    for idxs, stds in replicate_groups[:3]:
        print(f'  Rows {idxs}: target_std = {np.round(stds, 2)}')
if indeterminate_groups:
    print(f'Indeterminate groups (all targets <2 valid values, kept all rows): '
          f'{len(indeterminate_groups)} groups')
    for idxs in indeterminate_groups[:3]:
        print(f'  Rows {idxs} (kept as-is)')

Rows after merging: 494 (was 541)
Exact duplicates removed: 4 rows
  Row 404 -> removed Row 425
  Row 405 -> removed Row 426
  Row 406 -> removed Row 427
  Row 407 -> removed Row 428
Replicate groups merged (mean of targets): 19 groups
  Rows [501, 508, 515, 522]: target_std = [4.04 2.68 0.43]
  Rows [502, 509, 516, 523]: target_std = [4.45 4.67 0.42]
  Rows [503, 510, 517, 524]: target_std = [5.58 4.3  0.44]


## 3. Clean

In [5]:
n0 = len(df)

# Drop rows with NaN in features
df = df.dropna(subset=FEATURE_COLS)
print(f'Dropped {n0-len(df)} rows with NaN in features')

# Remove negative mass fractions
n0 = len(df)
for c in MASS_FRAC_COLS:
    df = df[df[c] >= 0]
print(f'Dropped {n0-len(df)} rows with negative mass fractions')

# Remove rows with non-positive targets (skip NaN -- keep missing for imputation)
for t in TARGET_COLS:
    n0 = len(df)
    valid = df[t].notna()
    df = df[~(valid & (df[t] <= 0))]
    print(f'Dropped {n0-len(df)} rows where {t} <= 0')

print(f'\nCleaned: {len(df)} rows')
print(f'Missing targets: YS={df["YS"].isna().sum()}  UTS={df["UTS"].isna().sum()}  El={df["El"].isna().sum()}')

Dropped 0 rows with NaN in features
Dropped 0 rows with negative mass fractions
Dropped 0 rows where YS <= 0
Dropped 0 rows where UTS <= 0
Dropped 0 rows where El <= 0

Cleaned: 494 rows
Missing targets: YS=49  UTS=15  El=37


## 4. Missing Value Report

In [6]:
N = len(df)
print(f'\n{"="*50}')
print(f'  MISSING VALUE REPORT ({N} rows)')
print(f'{"="*50}')
for t in TARGET_COLS:
    nm = df[t].isna().sum()
    print(f'  {t}: {nm}/{N} missing ({nm/N*100:.1f}%)')

# Both statistics
total_cells = df[TARGET_COLS].isna().sum().sum()
rows_with_any = df[TARGET_COLS].isna().any(axis=1).sum()
print(f'\n  Total missing cells:  {total_cells}  (values to impute by MICE)')
print(f'  Rows with any missing: {rows_with_any}  (samples with incomplete targets)')

# Missing combinations
df['_miss_pat'] = df[TARGET_COLS].apply(
    lambda r: 'complete' if r.isna().sum()==0 else ' + '.join(r.index[r.isna()])+' missing', axis=1)
for pat, cnt in df['_miss_pat'].value_counts().items():
    print(f'    {pat:40s} {cnt:>4} ({cnt/N*100:5.1f}%)')
df.drop(columns=['_miss_pat'], inplace=True)
print(f'{"="*50}\n')


  MISSING VALUE REPORT (494 rows)
  YS: 49/494 missing (9.9%)
  UTS: 15/494 missing (3.0%)
  El: 37/494 missing (7.5%)

  Total missing cells:  101  (values to impute by MICE)
  Rows with any missing: 72  (samples with incomplete targets)
    complete                                  422 ( 85.4%)
    YS missing                                 31 (  6.3%)
    YS + El missing                            14 (  2.8%)
    El missing                                 12 (  2.4%)
    UTS + El missing                           10 (  2.0%)
    YS + UTS missing                            3 (  0.6%)
    UTS missing                                 1 (  0.2%)
    YS + UTS + El missing                       1 (  0.2%)



## 5. Preliminary 70/30 Split (seed=42)

Split the full cleaned dataset into train (70%) and holdout (30%).
This split is ONLY for pipeline development (N02 feature generation, N03 feature selection, N04 Phase A model selection).
N04 Phase B uses the FULL dataset with 50 fresh random splits for rigorous evaluation.

MICE imputation follows in the next section: fit on train, apply to holdout (transform only).

In [7]:
# Prepare full cleaned dataset (NaN targets preserved, for N04 50-split evaluation)
df_full = df.copy()

# ── Preliminary split: 70% for pipeline development, 30% holdout ──
# This split is ONLY for developing the pipeline (imputer selection, feature selection).
# N04 will use the FULL dataset with 50 fresh random splits for rigorous evaluation.
from sklearn.model_selection import train_test_split

df_train, df_holdout = train_test_split(df_full, test_size=0.30, random_state=RNG_SEED)
print(f'Full dataset: {len(df_full)} samples')
print(f'  Train (for N02/N03 pipeline dev): {len(df_train)}')
print(f'  Holdout (untouched reference):     {len(df_holdout)}')
print(f'Missing targets in train — YS: {df_train["YS"].isna().sum()}  '
      f'UTS: {df_train["UTS"].isna().sum()}  El: {df_train["El"].isna().sum()}')

Full dataset: 494 samples
  Train (for N02/N03 pipeline dev): 345
  Holdout (untouched reference):     149
Missing targets in train — YS: 36  UTS: 8  El: 25


## 6. MICE Imputer Selection + Imputation（fit on train, transform both）

Imputer type selected via cv_rmse on the training set.
Best imputer fitted on train only → applied to both train (fit+transform) and holdout (transform only).
The imputer TYPE is saved to `best_imputer.json` for N04 per-split MICE.

- **Train**: fit + transform → fully imputed → saved to `metal_train_imputed.csv`
- **Holdout**: transform only (using fitted imputer) → fully imputed → saved to `metal_holdout_imputed.csv`
- **Full**: NaN preserved → saved to `metal_cleaned.csv` → N04 Phase B does per-split MICE

In [8]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import Ridge, LassoCV, ElasticNetCV
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.ensemble import (RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor)
from sklearn.model_selection import KFold

try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except ImportError:
    XGBRegressor = None; HAS_XGB = False
try:
    from lightgbm import LGBMRegressor
    HAS_LGBM = True
except ImportError:
    LGBMRegressor = None; HAS_LGBM = False

from featurize_metal import DNNRegressor

# ── 11 imputer candidates  ──
# LassoCV/EN included with internal CV for alpha/l1_ratio selection
ALPHAS = np.logspace(-4, 2, 50)
candidates = OrderedDict([
    # Built-in CV models (also serve as imputer estimators)
    ('LassoCV', LassoCV(alphas=ALPHAS, cv=5, random_state=RNG_SEED, max_iter=5000)),
    ('ElasticNetCV', ElasticNetCV(l1_ratio=[.1,.3,.5,.7,.9,.95,.99], cv=5,
                                 random_state=RNG_SEED, max_iter=5000)),
    # 9 models 
    ('Ridge', Ridge()),
    ('KNeighbors', KNeighborsRegressor(n_neighbors=5)),
    ('SVR', SVR()),
    ('RandomForest', RandomForestRegressor(n_estimators=100, random_state=RNG_SEED)),
    ('ExtraTrees', ExtraTreesRegressor(n_estimators=100, random_state=RNG_SEED)),
    ('GradientBoosting', GradientBoostingRegressor(n_estimators=100, random_state=RNG_SEED)),
] + ([('XGBoost', XGBRegressor(n_estimators=100, random_state=RNG_SEED, verbosity=0))] if HAS_XGB else [])
  + ([('LightGBM', LGBMRegressor(n_estimators=100, random_state=RNG_SEED, verbose=-1))] if HAS_LGBM else [])
  + [('DNN', DNNRegressor(random_state=RNG_SEED))])

print(f'{len(candidates)} imputer candidates')
print(f'Missing target cells in train set: {int(df_train[TARGET_COLS].isna().sum().sum())}')

11 imputer candidates
Missing target cells in train set: 69


In [9]:
# ── Step 1: Imputer selection via cv_rmse on TRAIN set only ──
# ── Step 2: Fit best imputer on train, apply to train + holdout   ──
# The imputer TYPE is saved for N04 per-split MICE;
# the imputed train data is used for N02/N03 pipeline development.

total_missing = int(df_train[TARGET_COLS].isna().sum().sum())
print(f'Total missing target values in train set: {total_missing}')

BEST_IMPUTER = None

if total_missing > 0:
    TARGET_WEIGHTS = np.array([0.4, 0.4, 0.2])

    def cv_rmse(estimator, X_feat, y_target, n_folds=5, missing_frac=0.1):
        rng = np.random.RandomState(RNG_SEED)
        kf = KFold(n_splits=n_folds, shuffle=True, random_state=RNG_SEED)
        y_std = np.nanstd(y_target, axis=0)
        y_std[y_std < 1e-9] = 1.0
        per_target_se = {i: [] for i in range(y_target.shape[1])}
        per_target_raw_se = {i: [] for i in range(y_target.shape[1])}
        per_target_true = {i: [] for i in range(y_target.shape[1])}
        per_target_pred = {i: [] for i in range(y_target.shape[1])}
        for tr, va in kf.split(X_feat):
            Xt, Xv = X_feat[tr], X_feat[va]
            yt, yv = y_target[tr].copy(), y_target[va].copy()
            known_mask = ~np.isnan(yv)
            if known_mask.sum() == 0: continue
            kr, kc = np.where(known_mask)
            n_mask = max(1, int(len(kr) * missing_frac))
            mask_idx = rng.choice(len(kr), size=n_mask, replace=False)
            true_vals = np.array([yv[kr[j], kc[j]] for j in mask_idx])
            col_idx = np.array([kc[j] for j in mask_idx])
            for j in mask_idx: yv[kr[j], kc[j]] = np.nan
            Xy = np.concatenate([np.concatenate([Xt, yt], 1), np.concatenate([Xv, yv], 1)], 0)
            imp = IterativeImputer(estimator=estimator, random_state=RNG_SEED, max_iter=10, tol=1e-3)
            yv_imputed = imp.fit_transform(Xy)[-len(yv):, -yv.shape[1]:]
            pred_vals = np.array([yv_imputed[kr[j], kc[j]] for j in mask_idx])
            sq_err = (true_vals - pred_vals) ** 2
            for ci in range(y_target.shape[1]):
                mask_ci = col_idx == ci
                if mask_ci.sum() > 0:
                    per_target_se[ci].extend((sq_err[mask_ci] / (y_std[ci] ** 2)).tolist())
                    per_target_raw_se[ci].extend(sq_err[mask_ci].tolist())
                    per_target_true[ci].extend(true_vals[mask_ci].tolist())
                    per_target_pred[ci].extend(pred_vals[mask_ci].tolist())
        per_target_nrmse = np.array([np.sqrt(np.mean(per_target_se[i])) if per_target_se[i] else np.nan for i in range(y_target.shape[1])])
        per_target_rmse  = np.array([np.sqrt(np.mean(per_target_raw_se[i])) if per_target_raw_se[i] else np.nan for i in range(y_target.shape[1])])
        per_target_mae   = np.array([np.mean(np.abs(np.array(per_target_true[i]) - np.array(per_target_pred[i]))) if per_target_true[i] else np.nan for i in range(y_target.shape[1])])
        per_target_r2    = np.array([1 - np.sum((np.array(per_target_true[i]) - np.array(per_target_pred[i]))**2) / max(np.sum((np.array(per_target_true[i]) - np.mean(per_target_true[i]))**2), 1e-9) if per_target_true[i] else np.nan for i in range(y_target.shape[1])])
        overall = np.mean(per_target_nrmse[~np.isnan(per_target_nrmse)])
        weighted = np.sum(TARGET_WEIGHTS * per_target_nrmse)
        return weighted, overall, per_target_rmse, per_target_r2, per_target_mae

    # Evaluate imputer candidates on TRAIN set only
    X_feat = df_train[FEATURE_COLS].values
    y_target = df_train[TARGET_COLS].values.astype(float)

    scores = {}
    for name, est in candidates.items():
        t0 = time()
        w_nrmse, overall, per_rmse, per_r2, per_mae = cv_rmse(est, X_feat, y_target)
        dt = time() - t0
        scores[name] = {'w_nrmse': w_nrmse, 'overall': overall,
                        'per_rmse': per_rmse, 'per_r2': per_r2, 'per_mae': per_mae}
        best_so_far = min(scores.values(), key=lambda v: v['w_nrmse'])['w_nrmse']
        marker = ' <- best' if w_nrmse == best_so_far else ''
        raw_str = '  '.join([f'{TARGET_COLS[i]}: R2={per_r2[i]:.3f} RMSE={per_rmse[i]:.1f} MAE={per_mae[i]:.1f}' for i in range(len(TARGET_COLS))])
        print(f'  {name:20s}  wNRMSE={w_nrmse:.4f}  ({dt:.1f}s){marker}')
        print(f'    {raw_str}')

    best_name = min(scores, key=lambda k: scores[k]['w_nrmse'])
    best_metrics = scores[best_name]
    BEST_IMPUTER = best_name
    print(f'\nSelected imputer: {best_name}')
    print(f'  Weighted NRMSE = {best_metrics["w_nrmse"]:.4f}')

    # ── Step 2: Fit best imputer on TRAIN, apply to BOTH train + holdout ──
    # Fit on train
    X_feat_tr = df_train[FEATURE_COLS].values
    y_tr = df_train[TARGET_COLS].values.astype(float)
    imp = IterativeImputer(estimator=candidates[best_name], random_state=RNG_SEED,
                           max_iter=10, tol=1e-3)
    Xy_tr = np.concatenate([X_feat_tr, y_tr], 1)
    Xy_tr_i = imp.fit_transform(Xy_tr)
    NT = len(TARGET_COLS)
    for i, col in enumerate(TARGET_COLS):
        df_train[col] = Xy_tr_i[:, -NT + i]

    # Transform holdout with SAME fitted imputer
    X_feat_ho = df_holdout[FEATURE_COLS].values
    y_ho = df_holdout[TARGET_COLS].values.astype(float)
    Xy_ho = np.concatenate([X_feat_ho, y_ho], 1)
    Xy_ho_i = imp.transform(Xy_ho)  # transform only — no fit!
    for i, col in enumerate(TARGET_COLS):
        df_holdout[col] = Xy_ho_i[:, -NT + i]

    nn_tr = df_train[TARGET_COLS].isna().sum().sum()
    nn_ho = df_holdout[TARGET_COLS].isna().sum().sum()
    print(f'  Train after imputation:    remaining NaN = {nn_tr} {"OK" if nn_tr == 0 else "FAIL"}')
    print(f'  Holdout after imputation:  remaining NaN = {nn_ho} {"OK" if nn_ho == 0 else "FAIL"}')

    # Save imputer SELECTION info for N04 (imputer TYPE + cv_rmse metrics)
    imputer_info = {
        'best_imputer': best_name,
        'w_nrmse': float(best_metrics['w_nrmse']),
        'per_target': {t: {'r2': float(best_metrics['per_r2'][i]),
                           'rmse': float(best_metrics['per_rmse'][i]),
                           'mae': float(best_metrics['per_mae'][i])}
                       for i, t in enumerate(TARGET_COLS)},
        'all_scores': {k: {'w_nrmse': float(v['w_nrmse'])} for k, v in scores.items()},
    }
    with open(os.path.join(DATA_DIR, 'best_imputer.json'), 'w') as f:
        json.dump(imputer_info, f, indent=2)
    print(f'  Saved best_imputer.json ({best_name} — for N04 per-split MICE)')
else:
    print('No missing values in train set - skipping imputation.')
    with open(os.path.join(DATA_DIR, 'best_imputer.json'), 'w') as f:
        json.dump({'best_imputer': None, 'note': 'No missing values'}, f)

Total missing target values in train set: 69
  LassoCV               wNRMSE=0.3359  (21.1s) <- best
    YS: R2=0.961 RMSE=28.5 MAE=23.5  UTS: R2=0.821 RMSE=35.8 MAE=19.0  El: R2=0.578 RMSE=4.3 MAE=3.1
  ElasticNetCV          wNRMSE=0.4184  (203.0s)
    YS: R2=0.932 RMSE=37.7 MAE=30.2  UTS: R2=0.812 RMSE=36.6 MAE=26.5  El: R2=0.144 RMSE=6.2 MAE=5.1
  Ridge                 wNRMSE=0.3893  (0.9s)
    YS: R2=0.942 RMSE=35.0 MAE=29.2  UTS: R2=0.830 RMSE=34.9 MAE=26.6  El: R2=0.283 RMSE=5.6 MAE=4.6
  KNeighbors            wNRMSE=0.3631  (9.1s)
    YS: R2=0.926 RMSE=39.4 MAE=28.8  UTS: R2=0.813 RMSE=36.5 MAE=24.5  El: R2=0.645 RMSE=4.0 MAE=2.8
  SVR                   wNRMSE=0.8727  (0.3s)
    YS: R2=0.122 RMSE=135.7 MAE=113.7  UTS: R2=0.174 RMSE=76.8 MAE=66.0  El: R2=0.106 RMSE=6.3 MAE=5.5
  RandomForest          wNRMSE=0.3408  (122.5s)
    YS: R2=0.929 RMSE=38.6 MAE=28.4  UTS: R2=0.886 RMSE=28.6 MAE=18.9  El: R2=0.587 RMSE=4.3 MAE=3.2
  ExtraTrees            wNRMSE=0.2702  (79.6s) <- best
   

In [10]:
# ── Save outputs ──
output_cols = FEATURE_COLS + TARGET_COLS
if 'Reference (APA)' in df_full.columns:
    output_cols = output_cols + ['Reference (APA)']

# 1) Imputed train set → for N02 feature engineering + N03 feature selection
df_train[output_cols].to_csv(os.path.join(DATA_DIR, 'metal_train_imputed.csv'), index=False)
print(f'Saved metal_train_imputed.csv ({len(df_train)} samples, imputed)')

# 2) Imputed holdout set → for quick dev-phase validation
df_holdout[output_cols].to_csv(os.path.join(DATA_DIR, 'metal_holdout_imputed.csv'), index=False)
print(f'Saved metal_holdout_imputed.csv ({len(df_holdout)} samples, imputed)')

# 3) Full cleaned dataset (NaN preserved) → for N04 50-split evaluation
df_full[output_cols].to_csv(os.path.join(DATA_DIR, 'metal_cleaned.csv'), index=False)
print(f'Saved metal_cleaned.csv ({len(df_full)} samples, NaN preserved)')
print(f'  Missing: YS={df_full["YS"].isna().sum()}  UTS={df_full["UTS"].isna().sum()}  El={df_full["El"].isna().sum()}')

print('\n  Pipeline: train(seed=42) fit imputer → transform train + holdout')
print('  N04 will use metal_cleaned.csv with per-split MICE for 50-split evaluation')

print('\nDone. Next: 02_feature_engineering.ipynb')

Saved metal_train_imputed.csv (345 samples, imputed)
Saved metal_holdout_imputed.csv (149 samples, imputed)
Saved metal_cleaned.csv (494 samples, NaN preserved)
  Missing: YS=49  UTS=15  El=37

  Pipeline: train(seed=42) fit imputer → transform train + holdout
  N04 will use metal_cleaned.csv with per-split MICE for 50-split evaluation

Done. Next: 02_feature_engineering.ipynb
